# Notebook 02: Taxonomic Harmonization

**Project:** Cancer Microbiome Comparative Analysis  
**Student:** Claire Chien  
**Description:** This notebook rolls up Colorectal species-level abundance to genus level, aligns all three cancer datasets to a common genus vocabulary, creates a unified combined matrix, applies prevalence filtering and CLR transformation, and saves harmonized data for downstream analysis.

**Prerequisites:** Run `01_data_loading_and_qc.ipynb` first to generate the intermediate CSV files in `Results/`.

In [ ]:
# ============================================================
# CELL 1 — Import Libraries and Configure Paths
# ============================================================
# This notebook converts the three separate datasets (loaded in Notebook 01) into
# ONE combined, harmonized abundance table that all future notebooks can use.

import pandas as pd         # tables and DataFrames
import numpy as np          # fast array math
import matplotlib           # plotting library
matplotlib.use('Agg')       # save plots to files (no pop-up windows)
import matplotlib.pyplot as plt
import seaborn as sns       # pretty statistical charts
import os                   # file path operations
import warnings
from scipy.stats import gmean  # gmean = geometric mean (used in the CLR transformation)
# Geometric mean of [a, b, c] = cube_root(a × b × c)

warnings.filterwarnings('ignore')  # hide minor warnings

# Mirror the same folder paths used in Notebook 01
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))  # go one level up from Notebooks/

DATA_DIR    = os.path.join(BASE_DIR, 'Data')       # original raw data
RESULTS_DIR = os.path.join(BASE_DIR, 'Results')    # intermediate CSV files
FIGURES_DIR = os.path.join(BASE_DIR, 'Figures')    # saved plots

os.makedirs(RESULTS_DIR, exist_ok=True)   # create Results/ if it doesn't exist
os.makedirs(FIGURES_DIR, exist_ok=True)   # create Figures/ if it doesn't exist

print(f'BASE_DIR    : {BASE_DIR}')
print(f'RESULTS_DIR : {RESULTS_DIR}')
print(f'FIGURES_DIR : {FIGURES_DIR}')
print('\nAll imports successful.')

In [ ]:
# ============================================================
# CELL 2 — Load Intermediate Data from Notebook 01
# ============================================================
# Notebook 01 saved six CSV files (three abundance + three metadata).
# We load them all here to begin the harmonization process.

# --- Abundance tables (rows = samples, columns = bacteria species or genera) ---
abund_colorectal = pd.read_csv(os.path.join(RESULTS_DIR, 'abund_colorectal.csv'), index_col=0)
abund_breast     = pd.read_csv(os.path.join(RESULTS_DIR, 'abund_breast.csv'),     index_col=0)
abund_prostate   = pd.read_csv(os.path.join(RESULTS_DIR, 'abund_prostate.csv'),   index_col=0)

# --- Metadata tables (rows = samples, columns = condition + cancer_type) ---
meta_colorectal = pd.read_csv(os.path.join(RESULTS_DIR, 'meta_colorectal.csv'), index_col=0)
meta_breast     = pd.read_csv(os.path.join(RESULTS_DIR, 'meta_breast.csv'),     index_col=0)
meta_prostate   = pd.read_csv(os.path.join(RESULTS_DIR, 'meta_prostate.csv'),   index_col=0)

print('=== Loaded DataFrames from Results/ ===')
print(f'  abund_colorectal : {abund_colorectal.shape}  (samples x species)')   # species-level
print(f'  abund_breast     : {abund_breast.shape}  (samples x genera)')        # already genus-level
print(f'  abund_prostate   : {abund_prostate.shape}  (samples x genera)')      # already genus-level
print(f'  meta_colorectal  : {meta_colorectal.shape}')
print(f'  meta_breast      : {meta_breast.shape}')
print(f'  meta_prostate    : {meta_prostate.shape}')

print('\nMetadata condition distribution:')
for name, meta in [('Colorectal', meta_colorectal), ('Breast', meta_breast), ('Prostate', meta_prostate)]:
    print(f'  {name}: {dict(meta["condition"].value_counts())}')

In [ ]:
# ============================================================
# CELL 3 — Species → Genus Rollup for the Colorectal Dataset
# ============================================================
# Colorectal columns look like "Bacteroides fragilis" (species level).
# Breast and Prostate already come at genus level, so we only roll up Colorectal.
#
# Steps:
#   1. Extract the first word of each species name to get the genus.
#   2. Group all species with the same genus and SUM their abundances.
#      (If Bacteroides fragilis = 5% and Bacteroides ovatus = 3%, then
#       genus Bacteroides = 8% in that sample.)
#   3. Transpose back so rows = samples and columns = genera.

n_species_before = abund_colorectal.shape[1]
print(f'Colorectal before rollup: {abund_colorectal.shape[0]} samples x {n_species_before} species')

def extract_genus(col_name):
    """
    Extract genus name from a species string by taking the first space-delimited word.
    'Bacteroides fragilis' → 'Bacteroides'
    'Escherichia coli K-12' → 'Escherichia'
    """
    parts = str(col_name).strip().split()   # split at whitespace
    return parts[0] if parts else col_name  # return first word; fallback = full name

# Show a sample of the species→genus mapping for inspection
genus_map   = {col: extract_genus(col) for col in abund_colorectal.columns}  # dict: species → genus
sample_items = list(genus_map.items())[:8]
print('\nSample species → genus mapping:')
for sp, gn in sample_items:
    print(f'  {sp!r:50s} -> {gn!r}')

# Transpose: rows become species names, columns become samples
abund_cr_T = abund_colorectal.T                      # now shape = (species x samples)
abund_cr_T.index = abund_cr_T.index.map(genus_map)  # rename rows: species → genus names

# Group rows by genus name and SUM (rollup: collapse multiple species into one genus)
# level=0 means "group by the row index" (which is now genus names)
abund_colorectal_genus = abund_cr_T.groupby(level=0).sum().T   # transpose back to samples x genera

n_genera_after = abund_colorectal_genus.shape[1]
print(f'\nColorectal after genus rollup : {abund_colorectal_genus.shape[0]} samples x {n_genera_after} genera')
print(f'  Species before rollup        : {n_species_before}')
print(f'  Compression ratio            : {n_species_before/n_genera_after:.1f}x')

# Row sums should still be close to 1.0 (relative abundance must sum to ~100%)
rs = abund_colorectal_genus.sum(axis=1)
print(f'\nRow-sum after rollup: min={rs.min():.4f}, max={rs.max():.4f}, mean={rs.mean():.4f}')

# Top 10 most abundant genera in Colorectal
top10 = abund_colorectal_genus.mean().sort_values(ascending=False).head(10)
print('\nTop 10 genera by mean relative abundance (Colorectal):')
for gn, val in top10.items():
    print(f'  {gn:30s}  {val:.5f}')

In [ ]:
# ============================================================
# CELL 4 — Find the Genus Vocabulary Across All Three Datasets
# ============================================================
# Each dataset used a slightly different sequencing pipeline, so they
# don't all detect the exact same genera. Before merging, we map out:
#   • Which genera appear in ALL three (intersection)
#   • Which genera appear in ANY of the three (union)
#   • Which are unique to just one dataset

genera_cr = set(abund_colorectal_genus.columns)   # all genera detected in Colorectal
genera_br = set(abund_breast.columns)             # all genera detected in Breast
genera_pr = set(abund_prostate.columns)           # all genera detected in Prostate

print('=== Genus vocabulary per dataset ===')
print(f'  Colorectal  : {len(genera_cr):5d} genera')
print(f'  Breast      : {len(genera_br):5d} genera')
print(f'  Prostate    : {len(genera_pr):5d} genera')

# UNION: all genera seen in at least one dataset — the "full vocabulary"
genera_union = genera_cr | genera_br | genera_pr   # | = set union in Python

# INTERSECTION: genera present in every dataset simultaneously
genera_intersection = genera_cr & genera_br & genera_pr  # & = set intersection

# Pairwise intersections (for reporting)
genera_cr_br = genera_cr & genera_br
genera_cr_pr = genera_cr & genera_pr
genera_br_pr = genera_br & genera_pr

# Genera UNIQUE to one dataset (not shared with any other)
only_cr = genera_cr - genera_br - genera_pr   # - = set difference
only_br = genera_br - genera_cr - genera_pr
only_pr = genera_pr - genera_cr - genera_br

print(f'\n  Union (any dataset)  : {len(genera_union):5d} genera')
print(f'  Intersection (all 3) : {len(genera_intersection):5d} genera')
print(f'  CR ∩ BR (pairwise)   : {len(genera_cr_br):5d} genera')
print(f'  CR ∩ PR (pairwise)   : {len(genera_cr_pr):5d} genera')
print(f'  BR ∩ PR (pairwise)   : {len(genera_br_pr):5d} genera')
print(f'\n  Unique to Colorectal : {len(only_cr):5d}')
print(f'  Unique to Breast     : {len(only_br):5d}')
print(f'  Unique to Prostate   : {len(only_pr):5d}')

print(f'\nStrategy: use the UNION ({len(genera_union)} genera) and fill missing values with 0.')
print('If a genus was not detected in a dataset, its abundance = 0 (not present).')
print('Prevalence filtering (Cell 7) will later remove very rare genera.')

union_cols = sorted(genera_union)   # sorted list for reproducible column order

In [ ]:
# ============================================================
# CELL 5 — Align All Three Datasets to the Union Vocabulary
# ============================================================
# reindex(columns=union_cols) does two things at once:
#   1. Adds any column from union_cols that is MISSING in this DataFrame (filled with 0).
#   2. Re-orders columns to match union_cols exactly (important for later comparison).
# fill_value=0.0 means "if a genus was not measured here, treat its abundance as 0."

abund_cr_aligned = abund_colorectal_genus.reindex(columns=union_cols, fill_value=0.0)
abund_br_aligned = abund_breast.reindex(           columns=union_cols, fill_value=0.0)
abund_pr_aligned = abund_prostate.reindex(         columns=union_cols, fill_value=0.0)

print('=== After alignment to union vocabulary ===')
print(f'  abund_cr_aligned : {abund_cr_aligned.shape}  (samples × {len(union_cols)} genera)')
print(f'  abund_br_aligned : {abund_br_aligned.shape}')
print(f'  abund_pr_aligned : {abund_pr_aligned.shape}')

# Verify all three matrices now have IDENTICAL column lists
assert list(abund_cr_aligned.columns) == list(abund_br_aligned.columns) == list(abund_pr_aligned.columns), \
    'Column mismatch after alignment!'
print(f'  Column counts match: {abund_cr_aligned.shape[1]} genera across all 3 matrices. OK.')

# Spot-check: a genus unique to Colorectal should have 0.0 in Breast and Prostate
if only_cr:
    ex_genus = next(iter(only_cr))   # pick any genus unique to Colorectal
    print(f'\n  Spot-check — genus unique to Colorectal: "{ex_genus}"')
    print(f'    Mean in Colorectal : {abund_cr_aligned[ex_genus].mean():.6f}')
    print(f'    Mean in Breast     : {abund_br_aligned[ex_genus].mean():.6f}  (should be 0.0)')
    print(f'    Mean in Prostate   : {abund_pr_aligned[ex_genus].mean():.6f}  (should be 0.0)')

# Row sums should still be ≤ 1.0 (relative abundance cannot exceed 100%)
for name, df in [('Colorectal', abund_cr_aligned), ('Breast', abund_br_aligned), ('Prostate', abund_pr_aligned)]:
    rs = df.sum(axis=1)
    print(f'  {name} row-sum range: [{rs.min():.4f}, {rs.max():.4f}]')

In [ ]:
# ============================================================
# CELL 6 — Stack All Three Datasets into One Combined Matrix
# ============================================================
# pd.concat stacks DataFrames vertically (axis=0 = row direction).
# After this step we have ONE big table with all 618 samples.

# --- Combine abundance tables (stack rows) ---
abund_combined = pd.concat(
    [abund_cr_aligned, abund_br_aligned, abund_pr_aligned],
    axis=0    # axis=0 = stack rows (as opposed to axis=1 which adds columns)
)

# --- Combine metadata (only keep columns that exist in all three) ---
meta_cols_shared = ['condition', 'cancer_type']
meta_combined = pd.concat(
    [
        meta_colorectal[meta_cols_shared],
        meta_breast[meta_cols_shared],
        meta_prostate[meta_cols_shared],
    ],
    axis=0
)

print('=== Combined matrix ===')
print(f'  abund_combined shape : {abund_combined.shape}  (all samples × all genera)')
print(f'  meta_combined shape  : {meta_combined.shape}')

# --- Check for duplicate sample IDs (could happen if datasets share IDs) ---
n_dup = abund_combined.index.duplicated().sum()
if n_dup > 0:
    print(f'\n  WARNING: {n_dup} duplicate sample IDs found — resolving by adding a suffix.')
    # Make duplicates unique by appending _1, _2, etc.
    seen = {}
    new_idx = []
    for idx in abund_combined.index:
        if idx in seen:
            seen[idx] += 1
            new_idx.append(f'{idx}_{seen[idx]}')
        else:
            seen[idx] = 0
            new_idx.append(idx)
    abund_combined.index = new_idx   # overwrite with unique IDs
    meta_combined.index  = new_idx   # keep metadata aligned to the same new IDs
    print('  -> Resolved by appending collision suffix.')
else:
    print(f'  Index is unique (no duplicate sample IDs). OK.')

# --- Verify that abundance and metadata rows are in the same order ---
assert list(abund_combined.index) == list(meta_combined.index), \
    'FATAL: abund_combined and meta_combined indices do not match!'
print('  abund_combined and meta_combined are aligned. OK.')

print('\nCondition breakdown in combined metadata:')
print(meta_combined.groupby(['cancer_type', 'condition']).size().to_string())

In [ ]:
# ============================================================
# CELL 7 — Prevalence Filtering (Remove Rare Genera)
# ============================================================
# Prevalence = fraction of samples where a genus has abundance > 0.
# A genus seen in only 1% of samples is likely noise or sequencing artifact.
# We keep only genera found in at least 10% of ALL 618 samples.
# This drops 1324 → 259 genera and removes mostly zeroes from our matrix.

PREVALENCE_THRESHOLD = 0.10   # 10% means the genus must appear in ≥ 62 of 618 samples

n_samples_total = abund_combined.shape[0]   # total number of samples
n_genera_before = abund_combined.shape[1]   # total genera before filtering

# For each column (genus), count the fraction of samples where it is > 0
prevalence = (abund_combined > 0).sum(axis=0) / n_samples_total
# prevalence is a Series; index = genus names, values = fraction of samples

# Keep only genera whose prevalence meets the threshold
keep_mask      = prevalence >= PREVALENCE_THRESHOLD   # boolean mask: True = keep this genus
abund_filtered = abund_combined.loc[:, keep_mask]     # filter columns (genera)
n_genera_after = abund_filtered.shape[1]

print(f'=== Prevalence filtering (threshold = {PREVALENCE_THRESHOLD:.0%}) ===')
print(f'  Genera before filtering : {n_genera_before}')
print(f'  Genera after filtering  : {n_genera_after}')
print(f'  Genera removed          : {n_genera_before - n_genera_after}  (too rare to be reliable)')

# Per-cancer-type prevalence check for surviving genera
print('\n  Per-cancer-type statistics for filtered genera:')
for ct in ['Colorectal', 'Breast', 'Prostate']:
    ct_idx   = meta_combined[meta_combined['cancer_type'] == ct].index
    ct_abund = abund_filtered.loc[ct_idx]
    ct_prev  = (ct_abund > 0).mean(axis=0)   # fraction of THAT cancer type's samples with this genus
    print(f'  {ct:12s}  mean prevalence={ct_prev.mean():.3f}  '
          f'median={ct_prev.median():.3f}  '
          f'genera present={(ct_prev > 0).sum()}')

# --- Figure: prevalence histogram before and after filtering ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left panel: all genera before filtering
axes[0].hist(prevalence.values, bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(PREVALENCE_THRESHOLD, color='red', linestyle='--',
                label=f'Threshold = {PREVALENCE_THRESHOLD:.0%}')
axes[0].set_xlabel('Prevalence (fraction of samples)', fontsize=11)
axes[0].set_ylabel('Number of Genera', fontsize=11)
axes[0].set_title('Genus Prevalence\n(before filtering)', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=10)
sns.despine(ax=axes[0])

# Right panel: genera that passed the filter
prevalence_filtered = (abund_filtered > 0).sum(axis=0) / n_samples_total
axes[1].hist(prevalence_filtered.values, bins=30, color='darkorange', edgecolor='white')
axes[1].set_xlabel('Prevalence (fraction of samples)', fontsize=11)
axes[1].set_ylabel('Number of Genera', fontsize=11)
axes[1].set_title('Genus Prevalence\n(after filtering)', fontsize=11, fontweight='bold')
sns.despine(ax=axes[1])

plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, 'fig01_prevalence_filter.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'\nSaved prevalence figure: {fig_path}')

In [ ]:
# ============================================================
# CELL 8 — CLR (Centered Log-Ratio) Transformation
# ============================================================
# Problem: microbiome abundance data is "compositional" — each row sums to 1.
# This creates a mathematical constraint that distorts correlation analysis.
#
# Solution: CLR transformation converts relative abundances into log-ratios,
# removing the compositional constraint.
#
# Formula:  CLR(x_i) = log( x_i / geometric_mean(x) )
#           where x_i is the abundance of genus i in a sample,
#           and geometric_mean(x) = (x_1 × x_2 × ... × x_n)^(1/n)
#
# After CLR: row sums ≈ 0 (by construction — verify in the check below).
# Zero abundances cause log(0) = undefined, so we add a tiny pseudocount first.

from scipy.stats import gmean   # gmean computes the geometric mean efficiently

def clr_transform(df, pseudocount=1e-6):
    """
    Apply CLR transformation to a samples × taxa DataFrame.

    Parameters
    ----------
    df          : pd.DataFrame — non-negative relative abundances
    pseudocount : float — small value added to every cell before taking log
                  (avoids log(0) = -infinity for absent genera)

    Returns
    -------
    pd.DataFrame — CLR-transformed values (same shape as input)
    """
    df_pseudo = df + pseudocount   # add pseudocount to every cell so no zeros remain

    # Geometric mean of each ROW = geometric mean across all genera for that sample
    gm = df_pseudo.apply(lambda row: gmean(row), axis=1)   # one value per sample

    # Divide each row by its geometric mean, then take the natural logarithm
    clr = df_pseudo.div(gm, axis=0).apply(np.log)
    # .div(gm, axis=0) means: divide row i by gm[i] (axis=0 = align along rows)
    # .apply(np.log) applies log element-wise to the entire DataFrame
    return clr

print('Applying CLR transformation ...')
abund_clr = clr_transform(abund_filtered)   # input: prevalence-filtered abundance table

print(f'  Input shape      : {abund_filtered.shape}  (raw relative abundances)')
print(f'  CLR output shape : {abund_clr.shape}  (log-ratio values)')

# --- Verification: CLR row sums should be very close to zero ---
clr_row_sums = abund_clr.sum(axis=1)
print(f'\n  CLR row-sum check (must be ≈ 0):')
print(f'    min={clr_row_sums.min():.2e}  max={clr_row_sums.max():.2e}  mean={clr_row_sums.mean():.2e}')

# --- Preview a small slice of the CLR matrix ---
print('\n  CLR values (first 3 samples × first 5 genera):')
print(abund_clr.iloc[:3, :5].round(4).to_string())

# --- Distribution plot of all CLR values ---
fig, ax = plt.subplots(figsize=(8, 4))
clr_vals = abund_clr.values.flatten()   # all 618×259 values as one flat array
ax.hist(clr_vals, bins=80, color='teal', edgecolor='none', alpha=0.8)
ax.axvline(0, color='red', linestyle='--', linewidth=1.2, label='CLR = 0 (geometric mean)')
ax.set_xlabel('CLR value', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Distribution of CLR-Transformed Values\n(all 618 samples × 259 genera)', fontsize=11, fontweight='bold')
ax.legend(fontsize=10)
sns.despine(ax=ax)
plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, 'fig01_clr_distribution.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'\nSaved CLR distribution figure: {fig_path}')

In [ ]:
# ============================================================
# CELL 9 — Save All Harmonized Data to Results/
# ============================================================
# Four files are saved here; downstream notebooks will load them as their starting point.
# File 1 (genus): raw relative abundances after filtering — used by NB03 alpha & NB04 beta
# File 2 (CLR):   CLR-transformed abundances — used by NB05 differential abundance & NB06 ML
# File 3 (meta):  combined sample labels — used by every downstream notebook
# File 4 (cr_genus): Colorectal genus-level table kept for audit/comparison purposes

# --- 1. Raw genus-level combined (after prevalence filtering) ---
abund_combined_genus_path = os.path.join(RESULTS_DIR, 'abund_combined_genus.csv')
abund_filtered.to_csv(abund_combined_genus_path)   # index=True preserves sample IDs as row labels

# --- 2. CLR-transformed combined ---
abund_clr_path = os.path.join(RESULTS_DIR, 'abund_combined_clr.csv')
abund_clr.to_csv(abund_clr_path)

# --- 3. Combined metadata ---
meta_combined_path = os.path.join(RESULTS_DIR, 'meta_combined.csv')
meta_combined.to_csv(meta_combined_path)

# --- 4. Colorectal genus-level table (supplementary audit trail) ---
abund_cr_genus_path = os.path.join(RESULTS_DIR, 'abund_colorectal_genus.csv')
abund_colorectal_genus.to_csv(abund_cr_genus_path)

# Print a summary of what was saved
print('=== Saved to Results/ ===')
saved = [
    ('abund_combined_genus.csv',   abund_filtered.shape,           'genus-level, prevalence-filtered'),
    ('abund_combined_clr.csv',     abund_clr.shape,                'CLR-transformed'),
    ('meta_combined.csv',          meta_combined.shape,            'sample labels'),
    ('abund_colorectal_genus.csv', abund_colorectal_genus.shape,   'Colorectal genus rollup (audit)'),
]
for fname, shape, desc in saved:
    print(f'  {fname:40s}  shape={shape}   [{desc}]')

print('\nAll harmonized data saved successfully.')

In [ ]:
# ============================================================
# CELL 10 — Summary Table: Key Statistics per Cancer Type
# ============================================================
# Print a human-readable overview of the harmonized dataset.

rows = []
for ct in ['Colorectal', 'Breast', 'Prostate']:
    ct_idx  = meta_combined[meta_combined['cancer_type'] == ct].index
    ct_meta = meta_combined.loc[ct_idx]
    ct_abund = abund_filtered.loc[ct_idx]   # prevalence-filtered genus-level abundance

    n_total   = len(ct_idx)
    n_cancer  = (ct_meta['condition'] == 'Cancer').sum()    # cancer samples
    n_healthy = (ct_meta['condition'] == 'Healthy').sum()   # healthy controls

    # How many of the 259 filtered genera have any non-zero mean in this cancer type?
    n_genera_nonzero = (ct_abund.mean(axis=0) > 0).sum()

    # Most abundant genus on average across all samples of this cancer type
    top_genus      = ct_abund.mean(axis=0).idxmax()          # name of the top genus
    top_genus_mean = ct_abund.mean(axis=0).max()             # its mean relative abundance
    median_top     = ct_abund[top_genus].median()            # median abundance across samples

    rows.append({
        'Cancer Type'             : ct,
        'N Total'                 : n_total,
        'N Cancer'                : n_cancer,
        'N Healthy'               : n_healthy,
        'N Genera (≥10% prev.)'   : n_genera_nonzero,
        'Top Genus'               : top_genus,
        'Mean Abund'              : round(top_genus_mean, 5),
        'Median Abund'            : round(median_top, 5),
    })

summary_df = pd.DataFrame(rows).set_index('Cancer Type')

print('=== Harmonization Summary ===')
print(summary_df.to_string())   # to_string() prints the full DataFrame without truncation

print('\n=== Combined Dataset Totals ===')
print(f'  Total samples       : {abund_filtered.shape[0]}')
print(f'  Total genera        : {abund_filtered.shape[1]}  (after prevalence ≥ 10% filter)')
print(f'  CLR matrix shape    : {abund_clr.shape}')

print('\nNotebook 02 — Taxonomic Harmonization — COMPLETE.')
print('Next: run 03_alpha_diversity.ipynb')